In [4]:
from datasets import load_dataset
from transformers import GPT2Tokenizer, GPT2LMHeadModel, Trainer, TrainingArguments
import torch

# 1. Check if GPU is available and set the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 2. Load and prepare the dataset
dataset = load_dataset("csv", data_files="./../Dataset/data1.csv")

# Split the dataset into train and validation sets
train_dataset = dataset["train"]

# 3. Load the GPT-2 model and tokenizer
model_name = "gpt2-large"  # Change to "gpt2-large" for a larger model
model = GPT2LMHeadModel.from_pretrained(model_name).to(device)  # Move the model to the selected device
tokenizer = GPT2Tokenizer.from_pretrained(model_name)

# GPT-2 tokenizer does not include padding by default
tokenizer.pad_token = tokenizer.eos_token

# 4. Preprocess the dataset
def preprocess_function(examples):
    # Combine question, context, and answer into a single string for GPT-2
    inputs = [f"question: {q} context: {c} answer: {a}" for q, c, a in zip(examples['question'], examples['context'], examples['answer'])]
    
    # Tokenize the combined string
    model_inputs = tokenizer(inputs, max_length=512, padding="max_length", truncation=True)
    
    # The labels should be the same as the input IDs for GPT-2's causal language modeling
    model_inputs["labels"] = model_inputs["input_ids"].copy()

    return model_inputs

# Apply preprocessing
tokenized_dataset = train_dataset.map(preprocess_function, batched=True)

# 5. Set up training arguments with adjustments for GPU memory (batch size, gradient accumulation)
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=16,
    save_steps=10_000,
    save_total_limit=2,
    logging_dir='./logs',
    logging_steps=500,
    evaluation_strategy="steps",  # Enable evaluation at regular intervals
    eval_steps=10_000,  # Match save_steps for compatibility
    load_best_model_at_end=True,  # Load best model after evaluation
    fp16=True,
)

# 6. Initialize the Trainer
# Split the dataset into training and evaluation sets
train_dataset = train_dataset.select(range(len(train_dataset) // 2))  # Example split
eval_dataset = train_dataset.select(range(len(train_dataset) // 2, len(train_dataset)))  # Example split

# Apply preprocessing to both train and eval datasets
tokenized_train_dataset = train_dataset.map(preprocess_function, batched=True)
tokenized_eval_dataset = eval_dataset.map(preprocess_function, batched=True)

# Initialize the Trainer with eval_dataset
trainer = Trainer(
    model=model,                         # the model to be trained
    args=training_args,                  # training arguments
    train_dataset=tokenized_train_dataset,  # training dataset
    eval_dataset=tokenized_eval_dataset,    # evaluation dataset
    tokenizer=tokenizer,                 # tokenizer for text processing
)

# 7. Train the model
trainer.train()

# 8. Save the fine-tuned model
model.save_pretrained('./fine_tuned_gpt2_large')
tokenizer.save_pretrained('./fine_tuned_gpt2_large')

# 9. Optionally, evaluate the model
results = trainer.evaluate(eval_dataset=tokenized_eval_dataset)

# Print evaluation results
print(results)


Using device: cuda


model.safetensors:   1%|          | 21.0M/3.25G [00:00<?, ?B/s]

Error while downloading from https://cdn-lfs.hf.co/gpt2-large/5f47f3e12f91cd33b662ce7e433b6150ad5512b5884a2cee961b50e9c3bbebce?response-content-disposition=inline%3B+filename*%3DUTF-8%27%27model.safetensors%3B+filename%3D%22model.safetensors%22%3B&Expires=1737543450&Policy=eyJTdGF0ZW1lbnQiOlt7IkNvbmRpdGlvbiI6eyJEYXRlTGVzc1RoYW4iOnsiQVdTOkVwb2NoVGltZSI6MTczNzU0MzQ1MH19LCJSZXNvdXJjZSI6Imh0dHBzOi8vY2RuLWxmcy5oZi5jby9ncHQyLWxhcmdlLzVmNDdmM2UxMmY5MWNkMzNiNjYyY2U3ZTQzM2I2MTUwYWQ1NTEyYjU4ODRhMmNlZTk2MWI1MGU5YzNiYmViY2U%7EcmVzcG9uc2UtY29udGVudC1kaXNwb3NpdGlvbj0qIn1dfQ__&Signature=V2iT5zqtgBmsYcJmz0RFmBgOr%7E7txAstAXIzUvs5aT8K0%7ESu-ZFwIPUKIHZIFBwfkNU855jeJ0Zdk0osR%7EoY%7EHrRD3xmmAe2K7zK2xFCtb-OZNby9id4VKzZWicj3XNGhGZCahY4mr6e87PHf7mYYu6YVPtgefxs0eFjPM3Jlqmk4hN09GHW%7EuJXS7ooR5-EU-mtZ74icMqqoJa%7EMuD2hN83Qm5pNoafiPh3NKoaA3hXdsWFzbsN8K4Udg4fG6QiFGRudn5IqYnnDtc0EgRLaI7AMjcjRMh0ER8%7EkOCeKqdNkRUPkbqYh1-cJfstPQXah8YviEVB7d2nyB913P0mjA__&Key-Pair-Id=K3RPWS32NSSJCE: HTTPSConnectionPool(host='cdn-lfs.

model.safetensors:  71%|#######   | 2.30G/3.25G [00:00<?, ?B/s]

Error while downloading from https://cdn-lfs.hf.co/gpt2-large/5f47f3e12f91cd33b662ce7e433b6150ad5512b5884a2cee961b50e9c3bbebce?response-content-disposition=inline%3B+filename*%3DUTF-8%27%27model.safetensors%3B+filename%3D%22model.safetensors%22%3B&Expires=1737543450&Policy=eyJTdGF0ZW1lbnQiOlt7IkNvbmRpdGlvbiI6eyJEYXRlTGVzc1RoYW4iOnsiQVdTOkVwb2NoVGltZSI6MTczNzU0MzQ1MH19LCJSZXNvdXJjZSI6Imh0dHBzOi8vY2RuLWxmcy5oZi5jby9ncHQyLWxhcmdlLzVmNDdmM2UxMmY5MWNkMzNiNjYyY2U3ZTQzM2I2MTUwYWQ1NTEyYjU4ODRhMmNlZTk2MWI1MGU5YzNiYmViY2U%7EcmVzcG9uc2UtY29udGVudC1kaXNwb3NpdGlvbj0qIn1dfQ__&Signature=V2iT5zqtgBmsYcJmz0RFmBgOr%7E7txAstAXIzUvs5aT8K0%7ESu-ZFwIPUKIHZIFBwfkNU855jeJ0Zdk0osR%7EoY%7EHrRD3xmmAe2K7zK2xFCtb-OZNby9id4VKzZWicj3XNGhGZCahY4mr6e87PHf7mYYu6YVPtgefxs0eFjPM3Jlqmk4hN09GHW%7EuJXS7ooR5-EU-mtZ74icMqqoJa%7EMuD2hN83Qm5pNoafiPh3NKoaA3hXdsWFzbsN8K4Udg4fG6QiFGRudn5IqYnnDtc0EgRLaI7AMjcjRMh0ER8%7EkOCeKqdNkRUPkbqYh1-cJfstPQXah8YviEVB7d2nyB913P0mjA__&Key-Pair-Id=K3RPWS32NSSJCE: HTTPSConnectionPool(host='cdn-lfs.

...

Map:   0%|          | 0/101 [00:00<?, ? examples/s]

C:\Users\kavin\AppData\Local\Temp\ipykernel_4428\1629503016.py:65: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


  0%|          | 0/18 [00:00<?, ?it/s]

{'train_runtime': 13323.9972, 'train_samples_per_second': 0.045, 'train_steps_per_second': 0.001, 'train_loss': 57.29576280381944, 'epoch': 2.63}


RuntimeError: CUDA error: CUBLAS_STATUS_EXECUTION_FAILED when calling cublasLtMatmul with transpose_mat1 0 transpose_mat2 0 m 3840 n 4096 k 1280 mat1_ld 3840 mat2_ld 1280 result_ld 3840 abcType 2 computeType 68 scaleType 0

In [ ]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel

# 1. Load the fine-tuned model and tokenizer
model_name_or_path = "./../fine_tuned_gpt2_large"
tokenizer = GPT2Tokenizer.from_pretrained(model_name_or_path)
model = GPT2LMHeadModel.from_pretrained(model_name_or_path)

# Ensure the pad token is set
tokenizer.pad_token = tokenizer.eos_token

# 2. Prepare the input
def prepare_input(question, context):
    """
    Prepare the input string in the same format as used during training.
    """
    input_text = f"question: {question} context: {context} answer:"
    return input_text

# question = "what supervised learninng?"
# context = "Supervised learning is a type of machine learning where the model is trained using labeled data. The goal is to teach the model to make predictions based on known outcomes."
question = "What is communication"
context = "Parallel systems involve multiple processors working on a single task simultaneously, often sharing memory and resources within a single system. An example is a supercomputer used for weather forecasting. Distributed systems, on the other hand, consist of independent systems working together over a network to solve problems, such as cloud platforms like AWS or Google Cloud. While parallel systems focus on speed and performance for a single task, distributed systems prioritize resource sharing and handling separate tasks."

input_text = prepare_input(question, context)

# Tokenize the input
input_ids = tokenizer(input_text, return_tensors="pt").input_ids

# 3. Generate the output
output_ids = model.generate(
    input_ids=input_ids,
    max_length=200,  # Adjust based on the expected answer length
    num_return_sequences=1,  # Number of outputs to generate
    temperature=0.7,  # Controls creativity
    top_k=50,  # Filters to top K likely tokens
    top_p=0.95,  # Nucleus sampling
    pad_token_id=tokenizer.eos_token_id,  # Ensure the model uses the correct pad token
)

# Decode the generated output
output_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)

# 4. Extract the answer
# The answer follows "answer:" in the generated text
answer = output_text.split("answer:")[-1].strip()

# 5. Print the result
print("Generated Answer:", answer)


C:\Users\kavin\AppData\Roaming\Python\Python312\site-packages\transformers\generation\configuration_utils.py:628: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
C:\Users\kavin\AppData\Roaming\Python\Python312\site-packages\transformers\generation\configuration_utils.py:633: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.95` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Generated Answer: Communication context: Parallel systems involve multiple processors working on a single task simultaneously, often sharing memory and resources within a single system. An example is a supercomputer used for weather forecasting. Distributed systems, on the other hand, consist of independent systems working together over a network to solve problems, such as cloud platforms like AWS or Google Cloud.


: 